# NeRF Inference on KCloud Jupyter

This notebook guides you through the entire NeRF assignment pipeline using a pre-trained checkpoint.

**Steps:**
1. Environment Setup (GPU check, dependency installation)
2. Implementation (Task 1 & Task 2 — edit source files as described in the README)
3. Rendering & Evaluation (Task 3)
4. Submission Packaging

## 1. Environment Setup

Please install `Python 3.12` and `pip`. Run `pip install -r requirements.txt` to install required packages on your KCloud environment.

### 1.1 GPU Check
Verify that CUDA device is available by running the following commands.

In [ ]:
!nvidia-smi

In [ ]:
# Verify PyTorch/CUDA environment
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Implementation

Complete the following two tasks before running the rendering cells below. Refer to the README for full details.

### Task 1. Implementing Ray Sampling

**Files to modify:**
- `torch_nerf/src/cameras/rays.py`
- `torch_nerf/src/renderer/ray_samplers/stratified_sampler.py`

**Sub-task 1.** Implement `compute_sample_coordinates` in `rays.py`.
For a ray $r$ with origin $\mathbf{o}$ and direction $\mathbf{d}$, a point on the ray is:

$$r(t) = \mathbf{o} + t\mathbf{d}, \quad t \in [t_n, t_f]$$

**Sub-task 2.** Implement `sample_along_rays_uniform` in `stratified_sampler.py`.
This performs stratified sampling (Eqn 2 in the paper):

$$t_i \sim \mathcal{U}\left[t_n + \frac{i-1}{N}(t_f - t_n),\; t_n + \frac{i}{N}(t_f - t_n)\right]$$

> 💡 See the helper functions `create_t_bins` and `map_t_to_euclidean`. `torch.rand_like` is useful for generating random samples.

### Task 2. Implementing Volume Rendering Equation

**File to modify:**
- `torch_nerf/src/renderer/integrators/quadrature_integrator.py`

Implement `integrate_along_rays`. This computes pixel color as a weighted sum of radiance along a ray (Eqn 3 in the paper):

$$\hat{C}(r) = \sum_{i=1}^{N} T_i \left(1 - \exp(-\sigma_i \delta_i)\right) \mathbf{c}_i, \quad T_i = \exp\left(-\sum_{j=1}^{i-1} \sigma_j \delta_j\right)$$

> 💡 `torch.exp`, `torch.cumsum`, and `torch.sum` are useful here.

## 3. Rendering & Evaluation (Task 3)

Render using the pre-trained checkpoint (`lego_ckpt.pth`).

⚠️ **Important**: Make sure you have completed **Task 1** (Ray Sampling) and **Task 2** (Volume Rendering Equation) above before running the cells below. The rendering will fail if these implementations are incomplete.

### 3.1 Render for Video (360-degree rotation)

In [ ]:
!PYTHONUNBUFFERED=1 python torch_nerf/runners/render.py +ckpt_path=./data/lego_ckpt.pth +render_test_views=False

In [ ]:
import glob
video_dirs = sorted(glob.glob('render/video/*'))
if video_dirs:
    VIDEO_IMG_DIR = video_dirs[-1]
    !python scripts/utils/create_video.py --img_dir {VIDEO_IMG_DIR} --vid_title nerf_lego
else:
    print("No video frames found.")

### 3.2 Render Test Views (for quantitative evaluation)

In [ ]:
!PYTHONUNBUFFERED=1 python torch_nerf/runners/render.py +ckpt_path=./data/lego_ckpt.pth +render_test_views=True

### 3.3 Visualize Rendered Images

In [ ]:
# Check rendered image paths
!ls -la render/

In [ ]:
# Visualize sample rendered images
import glob
from PIL import Image
from IPython.display import display

test_view_dirs = sorted(glob.glob('render/test_views/*'))
if test_view_dirs:
    latest_dir = test_view_dirs[-1]
    img_paths = sorted(glob.glob(f'{latest_dir}/*.png'), key=lambda x: int(x.split('_')[-1][:-4]))
    assert len(img_paths) == 100, "Expected 100 rendered images"
    imgs = [Image.open(p) for p in img_paths[::25]]
    w, h = imgs[0].size
    grid = Image.new('RGB', (w * 2, h * 2))
    for i, img in enumerate(imgs):
        grid.paste(img, (w * (i % 2), h * (i // 2)))
    display(grid)
else:
    print("No rendered images found.")

### 3.4 Quantitative Evaluation

Compute LPIPS and PSNR metrics on the test view renderings. Results are saved to `evaluation.txt`.

In [ ]:
import glob
test_view_dirs = sorted(glob.glob('render/test_views/*'))
if test_view_dirs and len(glob.glob(f'{(d := test_view_dirs[-1])}/*.png')) == 100:
    print(f"Rendering directory for evaluation: {d}")
    !PYTHONUNBUFFERED=1 python torch_nerf/runners/evaluate.py {d} ./data/nerf_synthetic/lego/test
else:
    print("No valid test renderings found (expected 100 images).")

## 4. Submission

Package source code, test renderings, and metrics into a ZIP file for KLMS submission. Enter your student ID when prompted.

In [ ]:
import os, glob, zipfile

STUDENT_ID = input("Enter your student ID: ").strip()
prefix = f"{STUDENT_ID}"

# Read metrics saved by evaluate.py
assert os.path.exists("evaluation.txt"), "evaluation.txt not found. Run the evaluation cell first."
metrics_str = open("evaluation.txt").read().strip()

# Find latest test renderings
test_view_dirs = sorted(glob.glob('render/test_views/*'))
assert test_view_dirs, "No test renderings found. Please run rendering first."
rendered_dir = test_view_dirs[-1]

# Build ZIP
zip_name = f"{prefix}.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 1) source code
    for root, dirs, files in os.walk('torch_nerf'):
        dirs[:] = [d for d in dirs if d != '__pycache__']
        for f in files:
            fpath = os.path.join(root, f)
            zf.write(fpath, os.path.join(prefix, fpath))

    # 2) Test view renderings
    for png in sorted(glob.glob(f'{rendered_dir}/*.png')):
        zf.write(png, os.path.join(prefix, f"{prefix}_renderings", os.path.basename(png)))

    # 3) Metrics text file
    zf.writestr(os.path.join(prefix, f"{prefix}.txt"), metrics_str)

print(f"Submission file created: {zip_name}")
print(f"  Total files: {len(zipfile.ZipFile(zip_name).namelist())}")